# XGBoost 전처리 체크리스트

XGBoost는 Tree 기반 모델이므로  
Linear / NN과 달리 스케일링, 정규성 등에 덜 민감하다.  
하지만 다음 전처리를 하면 성능이 안정적으로 좋아진다.


## 1. 기본 원칙

- EDA는 train 기준으로 수행
- 제거 / 변환 기준은 train 기준으로 결정
- test에는 동일하게 적용
- scaler / transformer는 train으로 fit 후 test transform

EDA를 train 기준으로 했고, test는 건드리면 안되지 않나?

👉 ❌ test도 똑같이 전처리 해야 함

👉 ✅ 기준은 train

👉 ✅ 적용은 train + test

train data만 전처리해놓으면
- leakage 발생
- distribution mismatch 발생
- 모델 성능 이상해짐

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# XGBoost (설치가 안 되어 있으면 아래 주석을 해제 후 실행)
# !pip -q install xgboost
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

plt.rcParams['font.family'] = 'Malgun Gothic' 
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 로드

이 프로젝트에서는 `product_type_1.csv`를 사용합니다.

이 CSV는 **2줄 헤더(MultiIndex)** 구조일 수 있습니다.
- 상위 레벨: process / sensor / defects
- 하위 레벨: velocity_1, bubble_1 등

그래서 `header=[0,1]`로 먼저 읽어보고,
- MultiIndex면 그대로 사용
- 아니면 일반 컬럼(flat)으로 처리합니다.


In [ ]:

try:
    df1 = pd.read_csv("../../data_processed/product_type_1.csv", header=[0, 1])
    multi = True
except Exception:
    df1 = pd.read_csv("../../data_processed/product_type_1.csv", header=[0, 1])
    multi = False

print("MultiIndex header:", multi)
print("Shape:", df1.shape)

if multi:
    print("Top-level columns:", df1.columns.levels[0].tolist())
else:
    print("Columns sample:", df1.columns[:15].tolist())

df1.head()


## 2. process / sensor / defects 블록 분리

- MultiIndex라면 `df1['process']` 처럼 접근 가능
- flat 컬럼이라면 `process_...`, `sensor_...`, `defects_...` 접두어로 찾아 분리

아래 코드는 두 경우를 모두 안전하게 처리합니다.


In [ ]:
def is_multiindex_columns(df: pd.DataFrame) -> bool:
    return isinstance(df.columns, pd.MultiIndex)

def get_block(df: pd.DataFrame, block_name: str) -> pd.DataFrame:
    if is_multiindex_columns(df):
        return df[block_name].copy()
    prefix = block_name.lower() + "_"
    cols = [c for c in df.columns if str(c).lower().startswith(prefix)]
    return df[cols].copy()

process_df = get_block(df1, "process")
sensor_df  = get_block(df1, "sensor")
defects_df = get_block(df1, "defects")

print("process_df:", process_df.shape)
print("sensor_df :", sensor_df.shape)
print("defects_df:", defects_df.shape)
print("\n[defects columns]")
print(defects_df.columns.tolist())


## 3. defect 그룹화 → multi-label 타겟(y) 만들기

우리가 예측할 y는 2개입니다.

- **표면**: dent/stain/exfoliation 중 하나라도 있으면 1
- **구조**: short_shot/bubble/deformation 중 하나라도 있으면 1

주의:
- defects 값이 0/1 뿐 아니라 2,3도 있을 수 있으므로
  - **0이면 양품**
  - **0초과면 불량(1로 통일)**
  로 이진화합니다.


In [ ]:
defects_numeric = defects_df.apply(pd.to_numeric, errors="coerce").fillna(0)
defects_bin = (defects_numeric > 0).astype(int)

defect_groups = {
    "표면": ["dent_1", "stain_1", "exfoliation_1", "exfoliation_2"],
    "구조": ["short_shot_1", "short_shot_2", "bubble_1", "bubble_2", "deformation_1", "deformation_2"],
}

def normalize_defect_colname(col):
    s = str(col).lower()
    return s.replace("defects_", "") if s.startswith("defects_") else s

map_to_real = {normalize_defect_colname(c): c for c in defects_bin.columns}

y = pd.DataFrame(index=df1.index)
for group_name, cols in defect_groups.items():
    real_cols = [map_to_real[c] for c in cols if c in map_to_real]
    if len(real_cols) == 0:
        raise KeyError(f"[{group_name}] 그룹에 해당하는 defects 컬럼을 찾지 못했습니다: {cols}")
    y[group_name] = defects_bin[real_cols].any(axis=1).astype(int)

print("[표면/구조 불량 비율(%)]")
display((y.mean() * 100).round(2).to_frame("rate_%"))
y.head()


## 4. X 만들기 + 데이터 누출(leakage) 방지

X는 **process + sensor**만 사용합니다.

그리고 아래는 “미래 예측에서 알 수 없거나”, “식별자/시간”이라서  
모델이 **공정 원인이 아니라 꼼수 패턴을 외우게** 만들 수 있습니다.

- process_id: 단순 식별번호
- process_shot: 생산 순서/카운터(시간 변수 성격)
- process_product_type: 이번 분석에서는 product_type_1만 사용 → 불필요/혼입 위험
- defect_flag_is_defect / is_defect: 이름부터 정답 힌트 가능

그래서 이런 컬럼을 **자동 탐지해서 제거**합니다.


In [ ]:
X = pd.concat([process_df, sensor_df], axis=1).copy()
X = X.drop(columns=["id","product_type","shot",'cylinder_pressure'], errors="ignore")
print("X shape (before drop):", X.shape)

def col_to_str(c):
    if isinstance(c, tuple):
        return "_".join([str(x) for x in c]).lower()
    return str(c).lower()

leak_keywords = [
    "process_id",
    "process_shot",
    "process_product_type",
    "defect_flag_is_defect",
    "is_defect",
]

leak_cols = [c for c in X.columns if any(k in col_to_str(c) for k in leak_keywords)]
if leak_cols:
    print("🚫 drop leakage cols:", leak_cols)
    X = X.drop(columns=leak_cols, errors="ignore")
else:
    print("✅ leakage 키워드 컬럼 없음")

defect_like = [c for c in X.columns if "defects" in col_to_str(c)]
if defect_like:
    print("🚫 drop defects-like cols from X:", defect_like)
    X = X.drop(columns=defect_like, errors="ignore")

print("X shape (after drop):", X.shape)


In [ ]:
print(X.info())

## 5. Train/Test 분리 (multi-label stratify)

y가 (표면, 구조) 2개 컬럼이므로 `stratify=y`를 그대로 쓰면 오류가 나거나 불안정할 수 있습니다.

그래서 (표면,구조) 조합을 문자열로 만들어 strata로 사용합니다.
- 00, 10, 01, 11 같은 형태


In [ ]:
strata = y.astype(str).agg("".join, axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strata
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

print("\n[라벨 조합  분포(전체)]")
display(strata.value_counts(normalize=True).sort_index().to_frame("ratio"))


# 단변량 분석
- 변수 하나의 분포/특성 파악


### 단변량 분석 시각화
- histogram = 분포 치우침, 이상치, 값 범위 확인
- Boxplot = 이상치 존재 여부, 분포 폭 확인


In [ ]:
print("=" * 60)
print("독립변수 분포 분석")
print("=" * 60)

import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

# 수치형 변수만 선택
num_cols = X_train.select_dtypes(include='number').columns

print(f"\n[수치형 변수 개수] {len(num_cols)}개")
print(num_cols.tolist())

# 각 변수의 기술통계 + 왜도 + 첨도 출력
summary_list = []

for col in num_cols:
    data = X_train[col].dropna()
    
    summary_list.append({
        '변수명': col,
        'count': data.count(),
        'mean': data.mean(),
        'std': data.std(),
        'min': data.min(),
        '25%': data.quantile(0.25),
        '50%': data.median(),
        '75%': data.quantile(0.75),
        'max': data.max(),
        '왜도': skew(data),
        '첨도': kurtosis(data)
    })

summary_df = pd.DataFrame(summary_list)
#print("\n[독립변수 통계 요약]")
#display(summary_df.round(3))

# 시각화
n_cols = 4   # 한 행에 변수 2개 (각 변수당 hist + box)
n_rows = math.ceil(len(num_cols) / (n_cols // 2))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

idx = 0

for col in num_cols:
    data = X_train[col].dropna()
    
    # 히스토그램
    sns.histplot(data, bins=30, ax=axes[idx], color='steelblue', kde=True)
    axes[idx].axvline(data.mean(), color='red', linestyle='--', linewidth=1.5,
                      label=f"mean: {data.mean():.2f}")
    axes[idx].axvline(data.median(), color='green', linestyle='--', linewidth=1.5,
                      label=f"median: {data.median():.2f}")
    axes[idx].set_title(f"{col} hist\n왜도={skew(data):.2f}, 첨도={kurtosis(data):.2f}", fontsize=12)
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

    # 박스플롯
    sns.boxplot(x=data, ax=axes[idx + 1], color='orange')
    axes[idx + 1].set_title(f"{col} box", fontsize=12)
    axes[idx + 1].grid(True, alpha=0.3)

    idx += 2

# 남는 subplot 제거
for j in range(idx, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# 독립변수 분포 분석 결과 해석

## 1. 전체 요약

독립변수 분포 시각화를 통해 다음과 같은 특징을 확인함.

- 왜도가 큰 변수가 다수 존재함
- 이상치가 많은 변수가 존재함
- 분산이 없는 변수(상수)가 존재함
- 일부 변수는 다봉분포(multimodal)를 보임
- 변수 간 스케일 차이가 매우 큼

따라서 모델 학습 전에 다음과 같은 전처리가 필요함.

- 분산이 거의 없는 변수 제거
- 왜도가 큰 변수에 대해 로그 변환
- 이상치에 강건한 스케일링 적용
- 전체 변수 스케일 정규화

---

## 2. 분산이 거의 없는 변수 (제거 후보)

다음과 같은 패턴이 관찰됨

- 히스토그램이 거의 한 줄로 나타남
- 박스플롯이 선 형태로 나타남
- 값의 변화가 거의 없음

예시 변수

- coolant_temp_min
- factory_temp_min
- factory_temp_max
- factory_humidity_min
- factory_humidity_max

이러한 변수는 모델에 정보량이 거의 없으므로 제거하는 것이 좋음.



In [ ]:
var_table = X_train.var().sort_values()

display(var_table.head(10))

print("="*60)
print("min,max는 모델에 의미 없는 변수 후보가 될 수도 있음")

## MIN MAX 들어가는 컬럼 제거하기



In [ ]:
cols_to_remove = [c for c in X_train.columns if ('min' in c) or ('max' in c)]

print("제거할 컬럼")
print(cols_to_remove)

X_train = X_train.drop(columns=cols_to_remove)
X_test  = X_test.drop(columns=cols_to_remove)


---

## 3. 왜도가 큰 변수 (로그 변환 후보)

다음과 같은 특징이 관찰됨

- 한쪽으로 치우친 분포
- 긴 꼬리(long tail)
- 박스플롯에서 이상치 다수 존재

예시 변수

- velocity_1
- velocity_2
- high_velocity
- rapid_rise_time

왜도가 큰 경우는 feature importance 안정화를 위해 로그 변환 또는 파워 변환이 필요하다.

→ skew > 3인 케이스를 로그로 변환

In [ ]:
skew_vals = X_train.skew().abs()

log_cols = skew_vals[skew_vals > 3].index

for c in log_cols:
    X_train[c] = np.log1p(X_train[c])
    X_test[c]  = np.log1p(X_test[c])

# 다변량 분석
- 변수 여러 개 간의 관계 분석

## 1. 다변량 분석의 목적

다변량 분석은 두 개 이상의 변수 간의 관계를 분석하여  
특정 변수(또는 타겟)에 영향을 주는 요인을 찾기 위한 과정이다.

본 데이터에서는 공정 변수(process), 센서 변수(sensor), 불량 정보(defects)를 이용하여  
표면 불량 및 구조 불량 발생과의 관계를 분석하는 것을 목표로 한다.

---

## 2. 데이터 구성

본 분석에서 사용된 데이터는 다음과 같이 구성된다.

| 데이터 | 설명 |
|--------|------|
| process_df | 공정 변수 |
| sensor_df | 센서 측정 변수 |
| defects_df | 불량 발생 정보 |
| y | 표면 / 구조 불량 여부 |
| X | 공정 + 센서 변수 |

---

## 3. 다변량 분석 방법

다변량 분석은 다음과 같은 형태로 수행한다.

### 3.1 그룹별 평균 비교

특정 그룹별로 수치형 변수의 중앙값 및 분포를 비교하여  
불량 발생과 관련된 변수를 찾는다.


목적

- 중요 변수 후보 탐색
- feature selection 참고

---

### 3.2 시각화 기반 비교

막대그래프 / 히스토그램 / 박스플롯을 이용하여  
변수 분포 차이를 시각적으로 확인한다.

사용한 그래프

- bar plot
- box plot
- histogram
- grouped bar plot

목적

- 분포 차이 확인
- 이상치 확인
- 평균 차이 확인

### 단변량 vs 다변량 핵심 차이

| 구분    | 단변량         | 다변량            |
| ----- | ----------- | -------------- |
| 변수 개수 | 1개          | 2개 이상          |
| 보는 것  | 분포          | 관계             |
| 목적    | 변수 자체 이해    | 변수 간 영향        |
| 질문    | 값이 어떻게 분포함? | 어떤 변수 때문에 달라짐? |




## EDA용 데이터 만들기 (편의를 위해..)

X_train과 y_train을 합쳐서, 분석용 데이터프레임 train_eda 생성

In [ ]:
print("\n" + "="*60)
print("EDA용 데이터 생성")
print("="*60)

train_eda = pd.concat(
    [X_train.reset_index(drop=True), y_train.reset_index(drop=True)],
    axis=1
)

print("train_eda shape:", train_eda.shape)
display(train_eda.head())
print("\n컬럼 목록:")
print(train_eda.columns.tolist())

## 통계적 가설 검정 (t-test / ANOVA)

EDA 과정에서 변수와 불량 발생 사이의 관계를 확인하기 위해
boxplot, 중앙값 비교, 구간별 집계 분석 등을 수행하였다.

하지만 이러한 시각적 비교만으로는 차이가 실제로 의미 있는지,
즉 통계적으로 유의한 차이인지 판단하기 어렵다.

따라서 각 공정 변수에 대해
불량 발생 여부에 따라 값의 평균이 실제로 다른지
통계적 검정을 수행한다.

---

## 1. t-test와 ANOVA의 목적

통계적 가설 검정의 목적은 다음과 같다.

- 특정 변수의 값이 그룹에 따라 차이가 있는지 확인
- 그 차이가 우연인지, 실제 차이인지 판단
- 모델에 사용할 중요한 변수 후보 찾기

즉,

> 변수 값의 차이가 불량 발생과 관련 있는지 검정한다.



---

## 2. EDA에서의 역할

EDA 과정에서 통계 검정은 다음 용도로 사용한다.

- 중요 변수 후보 찾기
- 불량과 관련된 변수 선별
- 모델 입력 변수 선택 보조
- 시각적 해석 검증

즉,

EDA → 통계검정 → 모델

순서로 진행한다.

---

## 3. 중앙값 vs 평균 문제와의 관계

이전에 이상치가 많아서 중앙값을 사용하였다.

하지만 t-test와 ANOVA는 평균 기반 검정이다.

그 이유는 다음과 같다.

- 통계 검정은 평균 차이를 기준으로 설계됨
- 불량 여부는 0/1 → 평균 = 비율
- 표본 수가 충분하면 평균 기반 검정 사용 가능

- `n_0`, `n_1` : 정상 / 불량 개수
- `mean_0`, `mean_1`: 정상군과 불량군의 평균
- `mean_diff`: 불량군 평균 - 정상군 평균
- `median_0`, `median_1`: 정상군과 불량군의 중앙값
- `median_diff`: 불량군 중앙값 - 정상군 중앙값
- `p_value`: 평균 차이의 통계적 유의성
- `significance`: 유의수준 표시

- `mean_0`, `mean_1`: 정상군과 불량군의 평균
- `mean_diff`: 불량군 평균 - 정상군 평균
- `median_0`, `median_1`: 정상군과 불량군의 중앙값
- `median_diff`: 불량군 중앙값 - 정상군 중앙값
- `p_value`: 평균 차이의 통계적 유의성
- `significance`: 유의수준 표시
---

## 4. 이번 분석에서 수행할 검정

이번 분석에서는 다음을 수행한다.

1. 각 공정 변수 vs 표면 불량 → t-test
2. 각 공정 변수 vs 구조 불량 → t-test
3. 변수 구간 vs 불량률 → ANOVA
4. 유의한 변수 찾기 (p-value 기준)

기준:

p-value < 0.05 → 유의함
p-value < 0.01 → 매우 유의함
p-value < 0.001 → 강한 관계


In [ ]:
from scipy.stats import ttest_ind

print("\n" + "="*60)
print("전체 변수 자동 t-test")
print("="*60)

# --------------------------------------------------
# 1. EDA용 데이터 준비
# --------------------------------------------------
train_eda = pd.concat(
    [X_train.reset_index(drop=True), y_train.reset_index(drop=True)],
    axis=1
).copy()

# 수치형 독립변수 추출
all_num_cols = train_eda.select_dtypes(include="number").columns.tolist()
target_cols = ["표면", "구조"]
feature_cols = [c for c in all_num_cols if c not in target_cols]

print(f"분석 대상 변수 개수: {len(feature_cols)}")
print(feature_cols)


# --------------------------------------------------
# 2. 단일 타겟에 대한 t-test 함수
# --------------------------------------------------
def run_ttest_all_features(df, feature_cols, target_col):
    results = []

    for col in feature_cols:
        temp = df[[col, target_col]].dropna().copy()

        # 그룹 분리
        g0 = temp[temp[target_col] == 0][col]
        g1 = temp[temp[target_col] == 1][col]

        # 그룹 크기 체크
        if len(g0) < 2 or len(g1) < 2:
            continue

        # Welch's t-test (등분산 가정 안 함)
        t_stat, p_val = ttest_ind(g0, g1, equal_var=False)

        mean_0 = g0.mean()
        mean_1 = g1.mean()
        median_0 = g0.median()
        median_1 = g1.median()

        results.append({
            "feature": col,
            "target": target_col,
            "n_0": len(g0),
            "n_1": len(g1),
            "mean_0": mean_0,
            "mean_1": mean_1,
            "mean_diff": mean_1 - mean_0,
            "median_0": median_0,
            "median_1": median_1,
            "median_diff": median_1 - median_0,
            "t_stat": t_stat,
            "p_value": p_val
        })

    result_df = pd.DataFrame(results)

    # 유의성 라벨
    def sig_label(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return ""

    result_df["significance"] = result_df["p_value"].apply(sig_label)
    result_df["is_significant_0.05"] = result_df["p_value"] < 0.05

    return result_df.sort_values("p_value", ascending=True).reset_index(drop=True)


# --------------------------------------------------
# 3. 표면 / 구조 각각 수행
# --------------------------------------------------
surface_ttest = run_ttest_all_features(train_eda, feature_cols, "표면")
structure_ttest = run_ttest_all_features(train_eda, feature_cols, "구조")

print("\n[표면 불량 기준 t-test 결과]")
display(surface_ttest)

print("\n[구조 불량 기준 t-test 결과]")
display(structure_ttest)


# --------------------------------------------------
# 4. 유의한 변수만 보기
# --------------------------------------------------
surface_sig = surface_ttest[surface_ttest["p_value"] < 0.05].copy()
structure_sig = structure_ttest[structure_ttest["p_value"] < 0.05].copy()

print("\n[표면 불량 기준 유의한 변수 (p < 0.05)]")
display(surface_sig)

print("\n[구조 불량 기준 유의한 변수 (p < 0.05)]")
display(structure_sig)


# --------------------------------------------------
# 5. 상위 변수만 보기
# --------------------------------------------------
top_n = 10

print(f"\n[표면 불량 기준 p-value 상위 {top_n}개 변수]")
display(surface_ttest.head(top_n))

print(f"\n[구조 불량 기준 p-value 상위 {top_n}개 변수]")
display(structure_ttest.head(top_n))

In [ ]:
print("\n" + "="*60)
print("T-test 결과에서 중요한 변수 추리기")
print("="*60)

# --------------------------------------------------
# 1. 표면 / 구조 유의 변수만 추출
# --------------------------------------------------
surface_t_sig = surface_ttest[surface_ttest["p_value"] < 0.05].copy()
structure_t_sig = structure_ttest[structure_ttest["p_value"] < 0.05].copy()

# --------------------------------------------------
# 2. 차이 큰 순으로 정렬
# mean_diff, median_diff 둘 다 고려
# --------------------------------------------------
surface_t_sig_sorted = surface_t_sig.sort_values(
    ["median_diff", "mean_diff", "p_value"],
    ascending=[False, False, True]
).reset_index(drop=True)

structure_t_sig_sorted = structure_t_sig.sort_values(
    ["median_diff", "mean_diff", "p_value"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("\n[표면 불량 기준 유의 변수]")
display(surface_t_sig_sorted)

print("\n[구조 불량 기준 유의 변수]")
display(structure_t_sig_sorted)


# --------------------------------------------------
# 3. 표면/구조에서 공통적으로 유의한 변수
# --------------------------------------------------
surface_sig_features = set(surface_t_sig_sorted["feature"])
structure_sig_features = set(structure_t_sig_sorted["feature"])

common_features = sorted(
    surface_sig_features & structure_sig_features
)

print("\n[표면/구조 공통적으로 유의한 변수]")
display(common_features)

## 결론 (t-test 기준)

### 표면 불량과 구조 불량 모두에서 유의하게 나타난 변수

- `air_pressure`
- `spray_time`
- `casting_pressure`
- `biscuit_thickness`
- `factory_humidity`

두 불량 모두에서 동일하게 유의하게 나타난 변수로,  
공정 압력, spray 조건, biscuit 두께, 환경 습도가  
불량 발생과 공통적으로 관련될 가능성이 있다.


### 표면 불량에서만 유의하게 나타난 변수

- `coolant_pressure`
- `pressure_rise_time`
- `velocity_2`
- `velocity_3`
- `coolant_temp`

표면 불량에서는 냉각 조건과 속도 조건 관련 변수에서  
정상군과 불량군 사이 차이가 나타났다.


### 구조 불량에서만 유의하게 나타난 변수

- `factory_temp`
- `high_velocity`
- `clamping_force`
- `spray_1_time`
- `rapid_rise_time`
- `velocity_1`
- `cycle_time`
- `melting_furnace_temp`

구조 불량에서는 환경 조건, spray 조건, 속도 조건,  
cycle 조건, 용탕 온도 관련 변수에서 차이가 나타났다.


### 종합 정리

- 두 불량 모두에서 공통적으로 차이가 나타난 변수  
  → 압력, 두께, spray, 습도 관련 변수

- 표면 불량은  
  → 냉각 조건, 속도 조건 영향이 함께 나타남

- 구조 불량은  
  → 환경 조건, spray 조건, cycle 조건, 온도 조건 영향이 크게 나타남

t-test 결과 기준으로 공통 핵심 변수는 다음과 같다.

**`biscuit_thickness`, `casting_pressure`, `air_pressure`, `factory_humidity`, `spray_time`**


##  표면 불량 기준 결과 해석

표면 불량 기준에서는 유의한 변수가 여러 개 확인되었지만,  
구조 불량에 비해 전반적으로 차이의 크기는 더 작게 나타났다.

즉,

> 표면 불량은 일부 공정 변수와 관련이 있으나,
> 구조 불량만큼 강한 패턴을 보이지는 않는다.

### 표면 불량과 관련 가능성이 높은 변수

- factory_humidity
- factory_temp
- casting_pressure
- biscuit_thickness
- coolant_temp
- air_pressure
- cylinder_pressure


근거

- 중앙값 차이 존재
- 평균 차이 존재
- p-value 유의
- 물리적으로 설명 가능

## 💣💣💣다중공선성 변수 다시 확인해봐야 함!! 밑에 적은 분석은 그 뒤에 다시 수정해야겠음


### 주요 유의 변수 상세 분석

#### `casting_pressure`
- `mean_diff = -11.88`
- `median_diff = -1`

해석

불량 제품에서 주조 압력이 더 낮은 경향을 보였다.

가능한 원인

- 충진 부족
- 기공 발생
- 표면 결함 증가

#### `biscuit_thickness`
- `mean_diff = -0.40`
- `median_diff = -1`

해석

두께가 얇을수록 표면 불량이 증가하는 경향

가능한 원인

- 금속 유동 부족
- 압력 부족
- 충진 불균형

#### `coolant_temp`
- `mean_diff = -0.25`
- `median_diff = -0.35`

해석

냉각수 온도가 낮을수록 표면 불량 증가 가능

가능한 원인

- 응고 속도 변화
- 표면 수축
- 열균열

#### `air_pressure`
- `mean_diff = +0.16`
- `median_diff = +0.10`

해석

에어 압력이 높을수록 불량 증가 가능

가능한 원인

- 분사 영향
- 냉각 영향
- 금형 상태 변화

#### `factory_humidity`
- `mean_diff = -1.74`
- `median_diff = -7.7`

해석

표면 불량이 발생한 경우 공장 습도가 더 낮은 경향을 보였다.

가능한 원인

- 냉각 조건 변화
- 금형 온도 영향
- 분사 조건 영향
- 산화 / 표면 상태 변화

### 해석에 주의가 필요한 변수

#### `spray_time`
- `p-value`는 매우 작음
- `median_diff = 0`

통계적으로는 유의하지만 중앙값 차이는 없다.  
즉, 실질적 차이보다 표본 수에 의해 유의성이 크게 나온 가능성이 있다.

#### `velocity_2`, `pressure_rise_time`
유의하긴 하지만 차이 크기가 매우 작다.  
모델에는 도움될 수 있으나, 공정 해석용 핵심 변수로 보기에는 약할 수 있다.


### ~~제외 또는 주의해야 할 변수~~

 #### ~~`shot`~~
~~`shot`은 순번처럼 보이지만 다시 1로 돌아가는 반복 구조를 가진다고 확인되었다.~~  
~~따라서 공정 조건 자체라기보다 순서, 배치, 운전 흐름의 영향을 반영했을 가능성이 있다.~~

~~즉,~~

~~통계적으로는 유의하더라도 공정 해석 변수로는 제외하는 것이 적절하다.~~

---

## 구조 불량 기준 결과 해석

구조 불량 기준에서는 유의한 변수가 더 많고,  
차이의 크기도 표면 불량보다 더 크게 나타났다.
그리고 구조 불량은 공정 변수뿐 아니라 환경 변수의 영향도 크게 받는 것으로 해석할 수 있다.


### 구조 불량과 관련 가능성이 높은 변수

- factory_humidity
- factory_temp
- casting_pressure
- biscuit_thickness
- melting_furnace_temp
- high_velocity

### 주요 유의 변수

#### `factory_humidity`
- `mean_diff = -7.03`
- `median_diff = -12.2`

해석

구조 불량이 발생한 경우 공장 습도가 정상보다 크게 낮은 경향을 보였다.  
평균과 중앙값 모두 큰 차이를 보여 구조 불량과 가장 강하게 관련된 변수 중 하나로 해석된다.

가능한 원인

- 냉각 조건 변화
- 금형 온도 변화
- 응고 속도 변화
- 표면 산화 및 내부 결함 증가

#### `factory_temp`
- `mean_diff = +1.60`
- `median_diff = +3.3`
해석

구조 불량이 발생한 경우 공장 온도가 정상보다 높은 경향을 보였다.  
습도와 함께 환경 조건이 구조 불량 발생에 영향을 줄 가능성이 크다.

가능한 원인

- 냉각 효율 저하
- 금형 온도 상승
- 응고 지연
- 내부 기공 발생 증가

#### `casting_pressure`
- `mean_diff = -13.92`
- `median_diff = 0`

해석

구조 불량이 발생한 경우 평균 기준 주조 압력이 더 낮게 나타났다.  
중앙값 차이는 크지 않지만 평균 차이가 크고 이전 분석 결과와도 방향이 일치하여 중요한 변수로 볼 수..있을까(애매)

가능한 원인

- 충진 부족
- 기공 발생 증가
- 금속 유동 불균형
- 내부 결함 증가

#### `biscuit_thickness`
- `mean_diff = -0.30`
- `median_diff = -1`

해석

구조 불량이 발생한 경우 biscuit thickness가 더 작은 경향을 보였다.  
표면 불량과 구조 불량 모두에서 차이가 나타나 공정 영향 가능성이 높은 변수이다.

가능한 원인

- 금속 유입 부족
- 압력 전달 부족
- 충진 불균형
- 내부 결함 증가

#### `melting_furnace_temp`
- `mean_diff = -3.90`
- `median_diff = -3.4`

해석

구조 불량이 발생한 경우 용해로 온도가 정상보다 낮게 나타났다.  

가능한 원인

- 용탕 온도 부족
- 유동성 저하
- 충진 불완전
- 내부 기공 발생

#### `high_velocity`
- `mean_diff = +0.003`
- `median_diff = +0.0019`

해석

구조 불량이 발생한 경우 고속 사출 속도가 약간 높은 경향을 보였다.  

가능한 원인

- 금속 흐름 불안정
- 난류 발생
- 기포 혼입 증가
- 내부 결함 발생

### 해석에 주의가 필요한 변수

#### `cycle_time`
- `p-value`는 매우 작음
- `median_diff = 0`

실제 공정 영향보다는 운영 조건 변화가 반영되었을 가능성이 있다.
- 공정 사이클 변화
- 냉각 시간 차이
- 생산 속도 변화

#### `spray_1_time`, `spray_2_time`, `pressure_rise_time`
유의하게 나타났으나 차이 크기는 작다.  
보조 변수로는 참고할 수 있으나, 핵심 공정 변수로 해석할 때는 신중해야 한다.


---

## 표면 불량과 구조 불량 비교

두 결과를 비교하면 다음과 같은 차이가 있다.

### 표면 불량
- 유의 변수는 존재하지만 차이 크기가 상대적으로 작음
- 중앙값 차이가 0인 변수도 많음
- 약한 신호가 여러 개 존재하는 형태

### 구조 불량
- 유의 변수 수가 더 많음
- 환경 변수(`factory_temp`, `factory_humidity`)의 영향이 매우 뚜렷함
- 압력 및 온도 관련 변수와의 관계도 더 분명함

즉,

> 구조 불량이 표면 불량보다 공정 변수와의 관련성이 더 강하게 나타났다.

---

##  최종 해석

이번 t-test 결과를 종합하면 다음과 같이 정리할 수 있다.

### 표면 불량에서 주목할 변수
- `casting_pressure`
- `biscuit_thickness`
- `coolant_temp`
- `air_pressure`
- `factory_humidity`

### 구조 불량에서 주목할 변수
- `factory_humidity`
- `factory_temp`
- `casting_pressure`
- `biscuit_thickness`
- `melting_furnace_temp`
- `high_velocity`

특히 구조 불량은 온도와 습도 같은 환경 변수와의 관계가 매우 강하게 나타났으며,  
표면/구조 공통으로는 `casting_pressure`, `biscuit_thickness`가 중요한 변수 후보로 보인다.

---

##  해석 시 주의사항

t-test 결과에서 `p-value`가 매우 작게 나오는 변수들이 많지만,  
이는 표본 수가 충분히 크기 때문에 아주 작은 차이도 유의하게 검출될 수 있기 때문이다.

따라서 변수 선택 시에는 다음을 함께 고려해야 한다.

- `p-value`
- `median_diff`
- boxplot 분포 차이
- 이전 EDA 결과와의 일관성
- 공정적으로 해석 가능한 변수인지 여부

~~특히 `shot`처럼 공정 순서성 변수는 통계적으로 유의해도~~
~~해석 변수로 채택하는 것은 적절하지 않을 수 있다.~~



=====================================================================

# 이번엔 ANOVA!
## ANOVA 분석 목적

t-test는 두 그룹의 평균 차이를 검정하는 방법이다.

이번 데이터에서는 불량 여부가 0/1이므로  
t-test를 통해 각 공정 변수와 불량 발생 사이의 평균 차이를 확인하였다.

하지만 공정 변수는 연속형 값이며,
특정 구간에서 불량률이 증가하거나 감소하는 패턴이 존재할 수 있다.

예를 들어

casting_pressure가

낮음 / 중간 / 높음 / 매우 높음

구간에 따라 불량률이 다를 수 있다.

이 경우 단순히 평균 비교(t-test)만으로는
관계를 충분히 확인하기 어렵다.

따라서 연속형 변수를 여러 구간으로 나눈 뒤

각 구간의 평균 또는 불량률이 서로 다른지

분산분석(ANOVA)을 수행한다.

---

## ANOVA의 목적

- 변수 값을 여러 구간으로 나눔
- 각 구간의 평균이 서로 다른지 검정
- 특정 구간에서 불량이 증가하는 패턴 확인

즉,

공정 변수 구간에 따라 불량률이 달라지는지 확인한다.


---

## 이번 분석에서 수행할 ANOVA

각 공정 변수에 대해

1. 값을 4구간으로 나눔 (quantile bin)
2. 각 구간의 불량 여부 평균 계산
3. 그룹 평균 차이에 대해 ANOVA 수행
4. p-value 기준으로 유의 변수 찾기

---

## 해석 기준

p-value < 0.05

→ 변수 구간에 따라 불량률이 다름

p-value 작고
구간별 불량률 차이도 크면

→ 중요한 변수 가능성 높음

### 컬럼명
- `feature`: 분석한 변수 이름
- `target`: 기준이 된 불량 유형(표면/구조)
- `n_bins`: 해당 변수를 나눈 구간 개수
- `f_stat`: ANOVA 검정 통계량
- `p_value`: 구간별 불량률 차이의 유의확률
- `min_rate`: 가장 낮은 구간 불량률
- `max_rate`: 가장 높은 구간 불량률
- `rate_diff`: 최고 불량률과 최저 불량률의 차이
- `bin_1 ~ bin_4`: 변수 값을 나눈 각 구간 범위
- `rate_1 ~ rate_4`: 각 구간에서의 불량률
- `significance`: p-value를 별표로 표시한 유의수준
- `is_significant_0.05`: 0.05 기준 유의 여부

In [ ]:
from scipy.stats import f_oneway

print("\n" + "="*60)
print("전체 변수 자동 ANOVA")
print("="*60)




# --------------------------------------------------
# 2. 단일 타겟에 대한 ANOVA 함수
# --------------------------------------------------
def run_anova_all_features(df, feature_cols, target_col, q=4):
    results = []

    for col in feature_cols:
        temp = df[[col, target_col]].dropna().copy()

        # qcut으로 구간화
        try:
            temp["bin"] = pd.qcut(temp[col], q=q, duplicates="drop")
        except ValueError:
            continue

        # 구간 수가 2개 미만이면 skip
        if temp["bin"].nunique() < 2:
            continue

        # 각 구간별 target 값 추출
        groups = [g[target_col].values for _, g in temp.groupby("bin", observed=False)]

        # 그룹 크기 체크
        if len(groups) < 2:
            continue
        if any(len(g) < 2 for g in groups):
            continue

        # ANOVA
        f_stat, p_val = f_oneway(*groups)

        # 구간별 불량률
        bin_rate = temp.groupby("bin", observed=False)[target_col].mean()

        result_row = {
            "feature": col,
            "target": target_col,
            "n_bins": temp["bin"].nunique(),
            "f_stat": f_stat,
            "p_value": p_val,
            "min_rate": bin_rate.min(),
            "max_rate": bin_rate.max(),
            "rate_diff": bin_rate.max() - bin_rate.min()
        }

        # 각 구간 불량률도 컬럼으로 저장
        for i, (bin_name, rate) in enumerate(bin_rate.items(), start=1):
            result_row[f"bin_{i}"] = str(bin_name)
            result_row[f"rate_{i}"] = rate

        results.append(result_row)

    result_df = pd.DataFrame(results)

    def sig_label(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return ""

    result_df["significance"] = result_df["p_value"].apply(sig_label)
    result_df["is_significant_0.05"] = result_df["p_value"] < 0.05

    return result_df.sort_values(["p_value", "rate_diff"], ascending=[True, False]).reset_index(drop=True)


# --------------------------------------------------
# 3. 표면 / 구조 각각 수행
# --------------------------------------------------
surface_anova = run_anova_all_features(train_eda, feature_cols, "표면", q=4)
structure_anova = run_anova_all_features(train_eda, feature_cols, "구조", q=4)

# print("\n[표면 불량 기준 ANOVA 결과]")
# display(surface_anova)

# print("\n[구조 불량 기준 ANOVA 결과]")
# display(structure_anova)


# --------------------------------------------------
# 4. 유의한 변수만 보기
# --------------------------------------------------
surface_anova_sig = surface_anova[surface_anova["p_value"] < 0.05].copy()
structure_anova_sig = structure_anova[structure_anova["p_value"] < 0.05].copy()

# print("\n[표면 불량 기준 유의한 변수 (p < 0.05)]")
# display(surface_anova_sig)

# print("\n[구조 불량 기준 유의한 변수 (p < 0.05)]")
# display(structure_anova_sig)


# --------------------------------------------------
# 5. 상위 변수만 보기
# --------------------------------------------------
top_n = 10

# print(f"\n[표면 불량 기준 p-value 상위 {top_n}개 변수]")
# display(surface_anova.head(top_n))

# print(f"\n[구조 불량 기준 p-value 상위 {top_n}개 변수]")
# display(structure_anova.head(top_n))

In [ ]:
print("\n" + "="*60)
print("ANOVA 결과에서 중요한 변수 추리기")
print("="*60)

# --------------------------------------------------
# 1. 표면 / 구조 유의 변수만 추출
# --------------------------------------------------
surface_anova_sig = surface_anova[surface_anova["p_value"] < 0.05].copy()
structure_anova_sig = structure_anova[structure_anova["p_value"] < 0.05].copy()

# --------------------------------------------------
# 2. rate_diff 큰 순으로 정렬
# --------------------------------------------------
surface_anova_sig_sorted = surface_anova_sig.sort_values(
    ["rate_diff", "p_value"],
    ascending=[False, True]
).reset_index(drop=True)

structure_anova_sig_sorted = structure_anova_sig.sort_values(
    ["rate_diff", "p_value"],
    ascending=[False, True]
).reset_index(drop=True)

print("\n[표면 불량 기준 유의 변수]")
display(surface_anova_sig_sorted)

print("\n[구조 불량 기준 유의 변수]")
display(structure_anova_sig_sorted)

# 3. 공통 유의 변수
common_features = sorted(
    set(surface_anova_sig["feature"]) &
    set(structure_anova_sig["feature"])
)

print("\n[표면/구조 공통적으로 유의한 변수]")
display(common_features)

## 결론

표면 불량과 구조 불량을 함께 보면 다음과 같이 정리할 수 있다.

### 공통적으로 의미 있게 나타난 변수
- `biscuit_thickness`
- `casting_pressure`
- `air_pressure`
- `factory_temp`
- `factory_humidity`
- `coolant_temp / coolant_pressure`
- `spray_time / spray_1_time`

### 표면 불량에서 특히 두드러진 변수
- `coolant_temp`
- `spray_1_time`
- `biscuit_thickness`
- `cycle_time`

### 구조 불량에서 특히 두드러진 변수
- `spray_time`
- `factory_temp`
- `factory_humidity`
- `spray_1_time`
- `cycle_time`

### 종합 정리
- 표면 불량은 **냉각 변수, spray 변수, 두께 변수, 압력 변수**의 영향이 크게 나타남
- 구조 불량은 **spray 변수, 환경 변수, cycle_time, 압력 변수**의 영향이 크게 나타남
- 두 불량 모두에서 **두께와 압력 계열 변수는 공통적으로 의미 있는 변수**로 나타남

